# PlantGuard AI preprocessing and split demonstration

The split CSVs are created by the reusable preparation script with a fixed seed. Exact-byte duplicate groups are excluded in full before stratification. Evaluation transforms perform resize, center crop, and normalization only. Stochastic augmentation is applied only to training data.


In [1]:
from pathlib import Path
import json
import pandas as pd
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


In [2]:
splits = {name: pd.read_csv(ROOT / f"dataset/splits/{name}.csv") for name in ("train", "val", "test")}
assert not (set(splits["train"].path) & set(splits["val"].path))
assert not (set(splits["train"].path) & set(splits["test"].path))
assert not (set(splits["val"].path) & set(splits["test"].path))
{key: len(value) for key, value in splits.items()}


{'train': 6621, 'val': 1419, 'test': 1419}

In [3]:
config = json.loads((ROOT / "models/preprocessing_config.json").read_text())
config


{'input_size': [224, 224],
 'color_mode': 'RGB',
 'resize': 'shorter edge to 256 pixels, then center crop 224x224 for evaluation',
 'normalization': {'mean': [0.485, 0.456, 0.406],
  'std': [0.229, 0.224, 0.225]},
 'channel_order': 'RGB',
 'augmentation_training_only': True,
 'seed': 42,
 'training_augmentation': ['RandomResizedCrop 224x224, scale 0.95-1.0, ratio 0.95-1.05',
  'RandomHorizontalFlip p=0.5',
  'ColorJitter brightness=0.10 contrast=0.10 saturation=0.08 hue=0.02']}

In [4]:
from src.plantguard.data import ArchiveImageDataset, make_loader, make_transform
from PIL import Image

# Custom transforms are accepted to support later robustness evaluation.
eval_transform = make_transform(training=False)
train_transform = make_transform(training=True)
class_to_idx = json.loads((ROOT / "models/class_mapping.json").read_text())["class_to_idx"]
train_data = ArchiveImageDataset(ROOT / "dataset/splits/train.csv", ROOT / "data/raw/Custom Dataset for Plant Species and Disease Detec/Ranveer_Realistic_dataset_2000_1126.zip", class_to_idx, transform=train_transform)
eval_data = ArchiveImageDataset(ROOT / "dataset/splits/val.csv", ROOT / "data/raw/Custom Dataset for Plant Species and Disease Detec/Ranveer_Realistic_dataset_2000_1126.zip", class_to_idx, transform=eval_transform)
train_loader = make_loader(train_data, batch_size=4, training=True, num_workers=0)
eval_loader = make_loader(eval_data, batch_size=4, training=False, num_workers=0)
train_batch, train_labels = next(iter(train_loader))
eval_batch, eval_labels = next(iter(eval_loader))
train_batch.shape, eval_batch.shape, train_labels.tolist(), eval_labels.tolist()


(torch.Size([4, 3, 224, 224]),
 torch.Size([4, 3, 224, 224]),
 [27, 2, 35, 2],
 [0, 0, 0, 0])

## Imbalance handling

Review `dataset/class_distribution.csv` and split counts before training. The reusable data module offers inverse-frequency class weights and a weighted sampler; the baseline CLI will expose these choices in Phase 2.
